# Notebook 5 — Topic Modeling: LDA + BERTopic
### Latent Dirichlet Allocation (Gensim) · BERTopic (sentence-transformers)

| Item | Detail |
|------|--------|
| **Input** | `Dataset/results/gold/gold_reviews.parquet` (6.99 M rows, 45 features) |
| **Task** | Unsupervised topic discovery from review text |
| **LDA sample** | 50 000 reviews — balanced across sentiment labels |
| **BERTopic sample** | 30 000 reviews |
| **Output** | Topic models + assignments saved to `Dataset/results/` |

**Goal:** Surface latent themes in Yelp reviews (e.g., *food quality*, *service*, *ambience*, *price*, *wait time*) and examine how topics distribute across sentiment labels (Negative / Neutral / Positive).

---
**Pipeline overview**
```
Gold Parquet ─► Sample ─┬─► Preprocess text ──► Bag-of-Words ──► LDA (Gensim)
                        │                                          └─ Coherence search (5-15 topics)
                        └─► Raw text ──► Sentence embeddings ──► UMAP ──► HDBSCAN ──► BERTopic
```

## 1 — Install Dependencies

In [1]:
!pip install gensim pyldavis bertopic sentence-transformers umap-learn hdbscan pyarrow nltk matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.5 MB/s eta 0:00:00


## 2 — Imports

In [2]:
import os, re, warnings, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# NLTK
import nltk
nltk.download(['stopwords', 'wordnet', 'punkt'], quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Gensim LDA
import gensim
from gensim import corpora
from gensim.models import LdaMulticore, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models

# BERTopic
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('gensim      :', gensim.__version__)
print('pyLDAvis    :', pyLDAvis.__version__)
print('All imports OK.')

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and sche

gensim      : 4.4.0
pyLDAvis    : 3.4.0
All imports OK.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4 — Paths & Directories

In [4]:
RESULTS_DIR = '/content/drive/MyDrive/Dataset/results'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
MODELS_DIR  = f'{RESULTS_DIR}/models'
PLOTS_DIR   = f'{RESULTS_DIR}/topic_plots'

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def save_fig(name):
    path = os.path.join(PLOTS_DIR, f'{name}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Saved → {path}')

print('Directories ready.')
print(f'  Models : {MODELS_DIR}')
print(f'  Plots  : {PLOTS_DIR}')

Directories ready.
  Models : /content/drive/MyDrive/Dataset/results/models
  Plots  : /content/drive/MyDrive/Dataset/results/topic_plots


In [5]:
import shutil, os

LOCAL_GOLD = '/content/gold_reviews.parquet'
GOLD_SRC   = f'{GOLD_DIR}/gold_reviews.parquet'

if not os.path.exists(LOCAL_GOLD):
    os.makedirs(LOCAL_GOLD, exist_ok=True)
    print('Copying gold parquet files to local disk ...')

    # Skip hidden .crc sidecar files — they trigger FUSE transport errors on Drive.
    # pyarrow and Spark readers do not need .crc files.
    files = sorted([
        f for f in os.listdir(GOLD_SRC)
        if not f.startswith('.') and (f.endswith('.parquet') or f == '_SUCCESS')
    ])
    print(f'  {len(files)} file(s) to copy.')
    for i, fname in enumerate(files, 1):
        shutil.copy2(os.path.join(GOLD_SRC, fname),
                     os.path.join(LOCAL_GOLD, fname))
        print(f'  [{i}/{len(files)}] {fname} ✓')
    print('Done.')
else:
    print('Local gold parquet already cached.')

Copying gold parquet files to local disk ...
  51 file(s) to copy.
  [1/51] _SUCCESS ✓
  [2/51] part-00000-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [3/51] part-00001-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [4/51] part-00002-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [5/51] part-00003-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [6/51] part-00004-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [7/51] part-00005-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [8/51] part-00006-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [9/51] part-00007-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [10/51] part-00008-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [11/51] part-00009-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [12/51] part-00010-24fa951c-a98a-4f21-a86d-37facb0cd06a-c000.snappy.parquet ✓
  [13/51] part-00011-24fa951c-a98a-4f21-a

## 5 — Load Gold Parquet
> Read only the four columns needed. `pyarrow` predicate push-down keeps memory low even on the 6.99 M-row file.

In [6]:
# Read only the four columns needed — from local disk (copied above to avoid Drive FUSE timeout)
gold_pq = pq.read_table(
    LOCAL_GOLD,
    columns=['text', 'stars', 'sentiment_label', 'sentiment_binary']
).to_pandas()

# Drop nulls and empty text
gold_pq = gold_pq.dropna(subset=['text', 'sentiment_label'])
gold_pq['text'] = gold_pq['text'].astype(str).str.strip()
gold_pq = gold_pq[gold_pq['text'].str.len() > 20].reset_index(drop=True)

print(f'Total usable reviews : {len(gold_pq):,}')
print('\nSentiment label distribution:')
print(gold_pq['sentiment_label'].value_counts().to_string())

Total usable reviews : 6,989,794

Sentiment label distribution:
sentiment_label
2    4684230
0    1613694
1     691870


## 6 — Sample
Balanced sampling across sentiment labels ensures topics are not dominated by the most common class.

In [7]:
LDA_SAMPLE  = 50_000
BERT_SAMPLE = 30_000

# --- LDA sample: balanced across sentiment_label ---
label_counts = gold_pq['sentiment_label'].value_counts()
n_labels     = len(label_counts)
per_label    = LDA_SAMPLE // n_labels

lda_parts = []
for label, grp in gold_pq.groupby('sentiment_label'):
    n = min(per_label, len(grp))
    lda_parts.append(grp.sample(n=n, random_state=42))
lda_df = pd.concat(lda_parts).sample(frac=1, random_state=42).reset_index(drop=True)

# --- BERTopic sample: stratified ---
per_label_bert = BERT_SAMPLE // n_labels
bert_parts = []
for label, grp in gold_pq.groupby('sentiment_label'):
    n = min(per_label_bert, len(grp))
    bert_parts.append(grp.sample(n=n, random_state=99))
bert_df = pd.concat(bert_parts).sample(frac=1, random_state=99).reset_index(drop=True)

print(f'LDA sample  : {len(lda_df):,} rows')
print(lda_df['sentiment_label'].value_counts().to_string())
print(f'\nBERTopic sample : {len(bert_df):,} rows')
print(bert_df['sentiment_label'].value_counts().to_string())

LDA sample  : 49,998 rows
sentiment_label
2    16666
0    16666
1    16666

BERTopic sample : 30,000 rows
sentiment_label
1    10000
0    10000
2    10000


---
# Part 1 — LDA (Latent Dirichlet Allocation)

LDA is a probabilistic generative model that represents each document as a mixture of topics and each topic as a distribution over words. We use **Gensim's `LdaMulticore`** for speed.

## 7 — Text Preprocessing

In [8]:
lemmatizer = WordNetLemmatizer()

# Extended stop-word list: English defaults + domain noise
STOP_WORDS = set(stopwords.words('english')) | {
    'would', 'could', 'get', 'also', 'one', 'go', 'make',
    'like', 'really', 'restaurant', 'food', 'place', 'time',
    'back', 'good', 'great', 'come', 'went', 'got', 'us',
    'even', 'much', 'well', 'way', 'still', 'first', 'two',
    'always', 'never', 'every', 'tried', 'came', 'told',
    'want', 'little', 'lot', 'going', 'made', 'said'
}

def preprocess(text):
    """Lowercase → strip non-alpha → remove stopwords → lemmatize."""
    text   = text.lower()
    text   = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in STOP_WORDS and len(t) > 2
    ]
    return tokens

print('Preprocessing LDA sample (50 k docs) ...')
processed_docs = lda_df['text'].apply(preprocess).tolist()
print(f'Done.  Example output (doc 0):')
print(processed_docs[0][:20])

Preprocessing LDA sample (50 k docs) ...
Done.  Example output (doc 0):
['delicious', 'service', 'slam', 'dunk', 'pretzel', 'amazing', 'mac', 'cheese', 'burger', 'absolutely', 'delicious', 'tom', 'server', 'man', 'top', 'refill', 'order', 'soon']


## 8 — Build Gensim Dictionary & Corpus

In [9]:
# Build dictionary
dictionary = corpora.Dictionary(processed_docs)
print(f'Raw vocabulary : {len(dictionary):,} tokens')

# Filter extremes:
#   no_below=10  → must appear in at least 10 docs
#   no_above=0.4 → must appear in at most 40% of docs
dictionary.filter_extremes(no_below=10, no_above=0.4)
print(f'Filtered vocab : {len(dictionary):,} tokens')

# Build bag-of-words corpus
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

# Quick stats
lengths = [len(doc) for doc in corpus]
print(f'\nCorpus : {len(corpus):,} documents')
print(f'  Avg tokens/doc : {np.mean(lengths):.1f}')
print(f'  Median         : {np.median(lengths):.1f}')
print(f'  Max            : {np.max(lengths)}')
print(f'\nSample BoW (doc 0, first 5 pairs):')
print([(dictionary[i], c) for i, c in corpus[0][:5]])

Raw vocabulary : 63,572 tokens
Filtered vocab : 10,230 tokens

Corpus : 49,998 documents
  Avg tokens/doc : 39.7
  Median         : 31.0
  Max            : 291

Sample BoW (doc 0, first 5 pairs):
[('absolutely', 1), ('amazing', 1), ('burger', 1), ('cheese', 1), ('delicious', 2)]


## 9 — Coherence Search (5 → 15 Topics)
Train LDA for each candidate number of topics and compute **c_v coherence** (higher is better). Pick the elbow.

In [10]:
NUM_TOPICS_RANGE = [5, 8, 10, 12, 15]
coherence_scores = []

print('Running coherence search ...')
print(f'{"Topics":>8}  {"c_v Coherence":>15}')
print('-' * 27)

for n in NUM_TOPICS_RANGE:
    tmp_model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=n,
        passes=5,           # fewer passes for search sweep
        workers=2,
        random_state=42,
        alpha='asymmetric',
        eta='auto'
    )
    cm = CoherenceModel(
        model=tmp_model,
        texts=processed_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f'{n:>8}  {score:>15.4f}')

best_idx       = int(np.argmax(coherence_scores))
BEST_NUM_TOPICS = NUM_TOPICS_RANGE[best_idx]
print(f'\nBest number of topics: {BEST_NUM_TOPICS}  (c_v = {coherence_scores[best_idx]:.4f})')

Running coherence search ...
  Topics    c_v Coherence
---------------------------
       5           0.4318
       8           0.5355
      10           0.5099
      12           0.5221
      15           0.5025

Best number of topics: 8  (c_v = 0.5355)


In [11]:
# Plot coherence vs. number of topics
plt.figure(figsize=(9, 5))
plt.plot(NUM_TOPICS_RANGE, coherence_scores, 'b-o', linewidth=2, markersize=8)
plt.axvline(x=BEST_NUM_TOPICS, color='red', linestyle='--', linewidth=1.5,
            label=f'Best = {BEST_NUM_TOPICS} topics')
plt.scatter([BEST_NUM_TOPICS], [coherence_scores[best_idx]],
            color='red', s=120, zorder=5)
plt.xlabel('Number of Topics', fontsize=12)
plt.ylabel('c_v Coherence Score', fontsize=12)
plt.title('LDA Coherence Score vs. Number of Topics', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.35)
plt.tight_layout()
save_fig('01_lda_coherence_search')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/01_lda_coherence_search.png


## 10 — Train Best LDA Model

In [12]:
# Use the best number of topics found above (default fallback = 10)
BEST_NUM_TOPICS = BEST_NUM_TOPICS if 'BEST_NUM_TOPICS' in dir() else 10

print(f'Training LDA with {BEST_NUM_TOPICS} topics, 10 passes ...')
lda_model = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=BEST_NUM_TOPICS,
    passes=10,
    workers=2,
    random_state=42,
    alpha='asymmetric',
    eta='auto'
)

# Final coherence of best model
final_cm = CoherenceModel(
    model=lda_model,
    texts=processed_docs,
    dictionary=dictionary,
    coherence='c_v'
)
final_coherence = final_cm.get_coherence()
print(f'\nFinal LDA c_v coherence : {final_coherence:.4f}')

Training LDA with 8 topics, 10 passes ...

Final LDA c_v coherence : 0.5451


## 11 — Inspect Topics

In [13]:
# Pretty-print top 10 words per topic
topic_data = []
print(f'Top words per topic  (LDA — {BEST_NUM_TOPICS} topics)\n' + '='*60)
for t_id in range(BEST_NUM_TOPICS):
    top_words = lda_model.show_topic(t_id, topn=10)
    words_str = ', '.join([w for w, _ in top_words])
    print(f'  Topic {t_id:>2} : {words_str}')
    topic_data.append({
        'topic_id'  : t_id,
        'top_words' : words_str,
        **{f'word_{i+1}': w for i, (w, _) in enumerate(top_words)},
        **{f'prob_{i+1}': round(p, 4) for i, (_, p) in enumerate(top_words)}
    })

# Save topics to CSV
topics_df = pd.DataFrame(topic_data)
topics_df.to_csv(f'{RESULTS_DIR}/lda_topics.csv', index=False)
print(f'\nTopics saved → {RESULTS_DIR}/lda_topics.csv')

Top words per topic  (LDA — 8 topics)
  Topic  0 : staff, store, dont, people, ive, love, friendly, price, location, nice
  Topic  1 : chicken, menu, service, ordered, salad, dish, meal, sauce, dinner, shrimp
  Topic  2 : order, service, minute, table, asked, drink, wait, didnt, customer, took
  Topic  3 : sandwich, burger, cheese, fry, coffee, breakfast, egg, cream, ordered, try
  Topic  4 : pizza, taco, sushi, roll, ordered, order, sauce, chicken, fresh, mexican
  Topic  5 : room, hotel, clean, stay, nice, floor, night, front, bathroom, area
  Topic  6 : car, day, service, call, called, customer, phone, company, work, week
  Topic  7 : nail, hair, cut, salon, color, done, job, massage, didnt, experience

Topics saved → /content/drive/MyDrive/Dataset/results/lda_topics.csv


## 12 — pyLDAvis Interactive Visualization

In [14]:
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim_models.prepare(
    lda_model, corpus, dictionary, sort_topics=False
)

# Save interactive HTML
html_path = f'{PLOTS_DIR}/lda_visualization.html'
pyLDAvis.save_html(vis, html_path)
print(f'Interactive LDA visualization saved → {html_path}')

# Display inline (works in Colab)
vis

Interactive LDA visualization saved → /content/drive/MyDrive/Dataset/results/topic_plots/lda_visualization.html


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.025307  0.065103       1        1  22.107796
1      0.160485 -0.011488       2        1  20.051176
2     -0.104661 -0.057277       3        1  17.956707
3      0.193608 -0.010386       4        1  10.904311
4      0.187675 -0.034783       5        1   6.926522
5     -0.092989  0.242088       6        1   5.849087
6     -0.205799 -0.072274       7        1  12.499172
7     -0.113013 -0.120983       8        1   3.705229, topic_info=          Term          Freq         Total Category  logprob  loglift
556       room   5686.000000   5686.000000  Default  30.0000  30.0000
1069     pizza   5426.000000   5426.000000  Default  29.0000  29.0000
306   sandwich   4514.000000   4514.000000  Default  28.0000  28.0000
2       burger   4351.000000   4351.000000  Default  27.0000  27.0000
7        order  11372.000000  11372.000000  Default  26.0000  26.0000
...        ...           ...           ...      ...      ...      ...
167       left    374.539729   3862.818281   Topic8  -5.4465   0.9620
11     service    471.152295  18587.678534   Topic8  -5.2170  -0.3796
172       nice    421.102317   9117.732992   Topic8  -5.3293   0.2203
933       work    360.879291   4583.760000   Topic8  -5.4837   0.7537
200       feel    344.720514   3585.426557   Topic8  -5.5295   0.9535

[619 rows x 6 columns], token_table=      Topic      Freq         Term
term                              
7864      3  0.953066      abysmal
2784      1  0.009671      account
2784      7  0.986432      account
4354      3  0.925323  acknowledge
4354      6  0.017681  acknowledge
...     ...       ...          ...
153       5  0.093282        youre
153       6  0.023878        youre
153       7  0.007641        youre
153       8  0.011461        youre
895       1  0.989670          zoo

[1589 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4, 5, 6, 7, 8])

## 13 — Topic Assignment
Assign each review in the LDA sample its dominant topic and record the associated probability.

In [15]:
def get_dominant_topic(bow):
    """Return (topic_id, probability) of the dominant topic for a BoW doc."""
    topic_dist = lda_model.get_document_topics(bow, minimum_probability=0)
    dominant   = max(topic_dist, key=lambda x: x[1])
    return dominant  # (topic_id, prob)

print('Assigning topics ...')
assignments = [get_dominant_topic(bow) for bow in corpus]

lda_df['topic_id']   = [a[0] for a in assignments]
lda_df['topic_prob'] = [a[1] for a in assignments]

print(f'Assignment complete.')
print('\nTopic frequency (top 5):')
print(lda_df['topic_id'].value_counts().head().to_string())
print(f'\nMean dominant-topic probability : {lda_df["topic_prob"].mean():.4f}')

Assigning topics ...
Assignment complete.

Topic frequency (top 5):
topic_id
0    11560
1    11061
2     8393
3     5934
6     5245

Mean dominant-topic probability : 0.6307


## 14 — Topic Analysis by Sentiment

In [16]:
# --- Bar chart: topic distribution by sentiment_label ---
ct = pd.crosstab(lda_df['topic_id'], lda_df['sentiment_label'])
ct_norm = ct.div(ct.sum(axis=0), axis=1)  # normalize per sentiment

# Define colour palette per sentiment
sent_palette = {'Negative': '#e74c3c', 'Neutral': '#f0ad4e', 'Positive': '#27ae60'}

fig, ax = plt.subplots(figsize=(14, 6))
ct.plot(kind='bar', ax=ax,
        color=[sent_palette.get(c, 'steelblue') for c in ct.columns],
        edgecolor='white', width=0.75, alpha=0.9)
ax.set_xlabel('Topic ID', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_title(f'LDA Topic Distribution by Sentiment Label ({BEST_NUM_TOPICS} Topics)',
             fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Sentiment', loc='upper right')
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
save_fig('02_lda_topic_distribution_by_sentiment')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/02_lda_topic_distribution_by_sentiment.png


In [17]:
# --- Heatmap: topic proportion per sentiment ---
# Rows = topics, Columns = sentiment labels, Values = fraction of that sentiment's docs in each topic
heatmap_data = ct_norm.T  # (sentiment x topic)

fig, ax = plt.subplots(figsize=(max(10, BEST_NUM_TOPICS * 0.9), 4))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.2%', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Proportion'}
)
ax.set_title('Topic Proportion per Sentiment Label (Heatmap)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Topic ID'); ax.set_ylabel('Sentiment Label')
plt.tight_layout()
save_fig('03_lda_topic_heatmap_by_sentiment')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/03_lda_topic_heatmap_by_sentiment.png


In [18]:
# --- Average topic probability by sentiment ---
mean_prob = lda_df.groupby(['sentiment_label', 'topic_id'])['topic_prob'].mean().unstack()

fig, ax = plt.subplots(figsize=(max(10, BEST_NUM_TOPICS * 0.9), 4))
sns.heatmap(
    mean_prob,
    annot=True, fmt='.3f', cmap='Blues',
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Mean Probability'}
)
ax.set_title('Mean Dominant-Topic Probability per Sentiment × Topic',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Topic ID'); ax.set_ylabel('Sentiment Label')
plt.tight_layout()
save_fig('04_lda_mean_prob_heatmap')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/04_lda_mean_prob_heatmap.png


## 15 — Save LDA Artifacts

In [19]:
# Save model
lda_model.save(f'{MODELS_DIR}/lda_model')
dictionary.save(f'{MODELS_DIR}/lda_dictionary.gensim')

# Save topic assignments
assignment_path = f'{RESULTS_DIR}/lda_topic_assignments.parquet'
lda_df[['text', 'stars', 'sentiment_label', 'topic_id', 'topic_prob']].to_parquet(
    assignment_path, index=False
)

# Save coherence summary
import json
lda_summary = {
    'best_num_topics'  : BEST_NUM_TOPICS,
    'final_coherence'  : round(final_coherence, 4),
    'coherence_sweep'  : dict(zip(NUM_TOPICS_RANGE, [round(s, 4) for s in coherence_scores])),
    'vocab_size'       : len(dictionary),
    'num_docs'         : len(corpus),
}
with open(f'{RESULTS_DIR}/lda_summary.json', 'w') as f:
    json.dump(lda_summary, f, indent=2)

print('LDA artifacts saved:')
print(f'  Model       → {MODELS_DIR}/lda_model')
print(f'  Dictionary  → {MODELS_DIR}/lda_dictionary.gensim')
print(f'  Assignments → {assignment_path}')
print(f'  Summary     → {RESULTS_DIR}/lda_summary.json')
print(json.dumps(lda_summary, indent=2))

LDA artifacts saved:
  Model       → /content/drive/MyDrive/Dataset/results/models/lda_model
  Dictionary  → /content/drive/MyDrive/Dataset/results/models/lda_dictionary.gensim
  Assignments → /content/drive/MyDrive/Dataset/results/lda_topic_assignments.parquet
  Summary     → /content/drive/MyDrive/Dataset/results/lda_summary.json
{
  "best_num_topics": 8,
  "final_coherence": 0.5451,
  "coherence_sweep": {
    "5": 0.4318,
    "8": 0.5355,
    "10": 0.5099,
    "12": 0.5221,
    "15": 0.5025
  },
  "vocab_size": 10230,
  "num_docs": 49998
}


---
# Part 2 — BERTopic (Neural Topic Modeling)

BERTopic uses **sentence-transformers** to embed documents, **UMAP** to reduce dimensionality, **HDBSCAN** to cluster, and class-based TF-IDF (c-TF-IDF) to extract topic representations. It is more semantically rich than bag-of-words LDA.

## 16 — BERTopic Setup

In [20]:
# Sentence embedding model (fast, high quality, 384-dim)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN for density-based clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer for c-TF-IDF representations
vectorizer_model = CountVectorizer(
    stop_words='english',
    min_df=10,
    ngram_range=(1, 2)   # unigrams + bigrams
)

# Build BERTopic pipeline
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True,
    nr_topics='auto'     # reduce via hierarchical merging
)

print('BERTopic pipeline configured.')
print(f'  Embedding model : all-MiniLM-L6-v2')
print(f'  UMAP components : 5')
print(f'  HDBSCAN min_cluster_size : 50')
print(f'  Vectorizer ngram_range   : (1, 2)')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BERTopic pipeline configured.
  Embedding model : all-MiniLM-L6-v2
  UMAP components : 5
  HDBSCAN min_cluster_size : 50
  Vectorizer ngram_range   : (1, 2)


## 17 — Fit BERTopic
> Embedding 30 k documents with `all-MiniLM-L6-v2` takes ~2 min on a T4 GPU.

In [21]:
bert_docs = bert_df['text'].tolist()

print(f'Fitting BERTopic on {len(bert_docs):,} documents ...')
topics, probs = topic_model.fit_transform(bert_docs)

topic_info = topic_model.get_topic_info()
n_topics   = topic_info.shape[0] - 1  # subtract outlier topic (-1)

print(f'\nNumber of topics found (excl. outlier): {n_topics}')
print(f'Outlier docs (topic -1)              : {sum(t == -1 for t in topics):,}')
print(f'Covered docs                          : {sum(t != -1 for t in topics):,}')
print('\nTop 15 topics by size:')
display(topic_info.head(16))

2026-05-14 14:32:13,665 - BERTopic - Embedding - Transforming documents to embeddings.


Fitting BERTopic on 30,000 documents ...


Batches:   0%|          | 0/938 [00:00<?, ?it/s]

2026-05-14 14:32:31,666 - BERTopic - Embedding - Completed ✓
2026-05-14 14:32:31,667 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-14 14:33:28,638 - BERTopic - Dimensionality - Completed ✓
2026-05-14 14:33:28,640 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-14 14:33:32,201 - BERTopic - Cluster - Completed ✓
2026-05-14 14:33:32,203 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-05-14 14:33:36,911 - BERTopic - Representation - Completed ✓
2026-05-14 14:33:36,912 - BERTopic - Topic reduction - Reducing number of topics
2026-05-14 14:33:36,938 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-14 14:33:41,516 - BERTopic - Representation - Completed ✓
2026-05-14 14:33:41,521 - BERTopic - Topic reduction - Reduced number of topics from 67 to 13



Number of topics found (excl. outlier): 12
Outlier docs (topic -1)              : 13,177
Covered docs                          : 16,823

Top 15 topics by size:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,13177,-1_food_good_service_place,"[food, good, service, place, just, time, like,...",[Food: 2 stars\nBeverages: 4 stars\nService: 3...
1,0,14113,0_good_food_place_just,"[good, food, place, just, like, great, service...",[1 for service and the decent plate that my au...
2,1,908,1_hair_time_did_appointment,"[hair, time, did, appointment, job, like, didn...",[I have been to Blowbar twice for a blowout wi...
3,2,420,2_store_items_shop_buy,"[store, items, shop, buy, like, prices, stuff,...","[The store has a lot of great products, but th..."
4,3,314,3_care_office_told_staff,"[care, office, told, staff, appointment, time,...",[I was very surprised to find that this busine...
5,4,256,4_dog_care_staff_time,"[dog, care, staff, time, took, know, said, app...",[My review would be five stars but for a few t...
6,5,197,5_equipment_month_member_room,"[equipment, month, member, room, like, people,...","[Oh, Planet Fitness. You're a good gym and a f..."
7,6,122,6_closed_open_hours_close,"[closed, open, hours, close, closing, door, sa...","[Door says it's open until 4 today, online say..."
8,7,115,7_car_job_clean_cleaning,"[car, job, clean, cleaning, paid, did, service...",[I came to this business on a day when everyon...
9,8,113,8_management_month_months_office,"[management, month, months, office, live, move...",[For the most part this place has been nothing...


## 18 — Visualize Top BERTopics

In [22]:
# Bar chart: top 15 topics by document count
# Exclude outlier topic (-1)
plot_info = topic_info[topic_info['Topic'] != -1].head(15).copy()

# Build label: Topic ID + top 3 words
def topic_label(row):
    words = topic_model.get_topic(row['Topic'])
    if not words:
        return f'Topic {row["Topic"]}'
    top3 = ', '.join([w for w, _ in words[:3]])
    return f'T{row["Topic"]}: {top3}'

plot_info['label'] = plot_info.apply(topic_label, axis=1)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    plot_info['label'][::-1],
    plot_info['Count'][::-1],
    color=plt.cm.tab20.colors[:len(plot_info)],
    edgecolor='white', alpha=0.9
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2,
            f'{int(w):,}', va='center', fontsize=9)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_title('BERTopic — Top 15 Topics by Size', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.35)
plt.tight_layout()
save_fig('05_bertopic_top15_bar')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/05_bertopic_top15_bar.png


In [23]:
# Word-score bar charts for top 6 topics
top6_ids = plot_info['Topic'].tolist()[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BERTopic — Top 6 Topic Word Distributions', fontsize=14, fontweight='bold')

for ax, t_id in zip(axes.flatten(), top6_ids):
    words_scores = topic_model.get_topic(t_id)
    if not words_scores:
        ax.axis('off')
        continue
    words  = [w for w, _ in words_scores[:10]]
    scores = [s for _, s in words_scores[:10]]
    ax.barh(words[::-1], scores[::-1],
            color=plt.cm.tab20.colors[top6_ids.index(t_id)], alpha=0.85)
    ax.set_title(f'Topic {t_id}', fontweight='bold')
    ax.set_xlabel('c-TF-IDF Score')
    ax.grid(axis='x', alpha=0.35)

plt.tight_layout()
save_fig('06_bertopic_word_scores')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/06_bertopic_word_scores.png


In [24]:
# BERTopic distribution by sentiment
bert_df['bertopic_id'] = topics

# Focus on non-outlier topics
bert_non_outlier = bert_df[bert_df['bertopic_id'] != -1]

# Top 10 topics only (for readability)
top10_ids_set = set(plot_info['Topic'].tolist()[:10])
bt_sub = bert_non_outlier[bert_non_outlier['bertopic_id'].isin(top10_ids_set)]

ct_bert = pd.crosstab(bt_sub['bertopic_id'], bt_sub['sentiment_label'])

fig, ax = plt.subplots(figsize=(14, 6))
ct_bert.plot(
    kind='bar', ax=ax,
    color=[sent_palette.get(c, 'steelblue') for c in ct_bert.columns],
    edgecolor='white', width=0.75, alpha=0.9
)
ax.set_xlabel('BERTopic ID', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_title('BERTopic — Top 10 Topics by Sentiment Label', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Sentiment', loc='upper right')
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
save_fig('07_bertopic_distribution_by_sentiment')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/07_bertopic_distribution_by_sentiment.png


## 19 — Save BERTopic Artifacts

In [25]:
# Save model
topic_model.save(f'{MODELS_DIR}/bertopic_model')

# Save topic info CSV
topic_info_path = f'{RESULTS_DIR}/bertopic_topic_info.csv'
topic_info.to_csv(topic_info_path, index=False)

# Save per-document assignments
bert_assign_path = f'{RESULTS_DIR}/bertopic_assignments.parquet'
bert_df[['text', 'stars', 'sentiment_label', 'bertopic_id']].to_parquet(
    bert_assign_path, index=False
)

# Save BERTopic summary
bert_summary = {
    'n_topics_found'    : int(n_topics),
    'n_outlier_docs'    : int(sum(t == -1 for t in topics)),
    'n_covered_docs'    : int(sum(t != -1 for t in topics)),
    'embedding_model'   : 'all-MiniLM-L6-v2',
    'umap_components'   : 5,
    'hdbscan_min_cluster': 50,
    'total_docs'        : len(bert_docs),
}
with open(f'{RESULTS_DIR}/bertopic_summary.json', 'w') as f:
    json.dump(bert_summary, f, indent=2)

print('BERTopic artifacts saved:')
print(f'  Model       → {MODELS_DIR}/bertopic_model')
print(f'  Topic info  → {topic_info_path}')
print(f'  Assignments → {bert_assign_path}')
print(f'  Summary     → {RESULTS_DIR}/bertopic_summary.json')
print(json.dumps(bert_summary, indent=2))

2026-05-14 14:33:43,705 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


BERTopic artifacts saved:
  Model       → /content/drive/MyDrive/Dataset/results/models/bertopic_model
  Topic info  → /content/drive/MyDrive/Dataset/results/bertopic_topic_info.csv
  Assignments → /content/drive/MyDrive/Dataset/results/bertopic_assignments.parquet
  Summary     → /content/drive/MyDrive/Dataset/results/bertopic_summary.json
{
  "n_topics_found": 12,
  "n_outlier_docs": 13177,
  "n_covered_docs": 16823,
  "embedding_model": "all-MiniLM-L6-v2",
  "umap_components": 5,
  "hdbscan_min_cluster": 50,
  "total_docs": 30000
}


---
# Part 3 — LDA vs BERTopic Comparison

| Dimension | LDA (Gensim) | BERTopic |
|-----------|-------------|----------|
| **Approach** | Probabilistic generative model | Neural embedding + clustering |
| **Text representation** | Bag-of-Words (sparse) | Sentence embeddings (dense, 384-dim) |
| **Topic count** | Must be set in advance | Determined automatically by HDBSCAN |
| **Word relationships** | Co-occurrence only | Semantic similarity via transformers |
| **Topic representation** | Top-N words by probability | c-TF-IDF on cluster docs; supports bigrams |
| **Outlier handling** | Every doc assigned to a topic | Topic -1 flags documents that don't cluster |
| **Interpretability** | High — explicit word-probability distributions | Moderate — words + representative documents |
| **Scalability** | Scales well (sparse BoW) | GPU recommended for >50 k docs |
| **Speed (50 k docs)** | ~5 min (CPU, 10 passes) | ~3 min (GPU, embeddings) |
| **Strengths** | Transparent, probabilistic, tunable | Captures nuance, handles polysemy, bigrams |
| **Weaknesses** | Ignores word order and semantics | Black-box embeddings; UMAP non-deterministic |

**Practical take-away:** LDA is preferred when interpretability and reproducibility are paramount. BERTopic is preferred when the text is linguistically rich, topics overlap semantically, or you need automatic topic count selection.

## 20 — Topic Quality Comparison Plot

In [26]:
# Side-by-side topic size distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Topic Modeling — Size Distributions', fontsize=14, fontweight='bold')

# LDA: topic size = number of docs where this is dominant topic
lda_sizes = lda_df['topic_id'].value_counts().sort_index()
axes[0].bar(lda_sizes.index.astype(str), lda_sizes.values,
            color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Topic ID')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_title(f'LDA ({BEST_NUM_TOPICS} topics, 50 k docs)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.35)

# BERTopic: top 20 topics by size (excluding outlier)
bt_sizes = topic_info[topic_info['Topic'] != -1].head(20)
axes[1].bar(bt_sizes['Topic'].astype(str), bt_sizes['Count'],
            color='darkorange', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Topic ID')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_title(f'BERTopic ({n_topics} topics, 30 k docs)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.35)

plt.tight_layout()
save_fig('08_lda_vs_bertopic_sizes')

  Saved → /content/drive/MyDrive/Dataset/results/topic_plots/08_lda_vs_bertopic_sizes.png
